# Optimal Tuning

This tutorial demonstrates how to use ORCA together with the ORCA python interface (OPI) to perform an optimal tuning procedure.

## Step 1: Import Dependencies

We start by importing the modules needed for:
- Interfacing with ORCA input/output
- Numerical calculations and data handling
- Plotting results

> **Note:** We additionally import modules for visualization/plotting like `py3Dmol`. For this, it might be necessary to install `py3Dmol` into your OPI `venv` (e.g., by activating the `.venv` and using `uv pip install py3Dmol`).

In [1]:
from pathlib import Path
import shutil
import copy
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from scipy.optimize import minimize_scalar


# > pandas and numpy for data handling
import pandas as pd
import numpy as np

# > OPI imports for performing ORCA calculations and reading the output
from opi.core import Calculator
from opi.output.core import Output
from opi.input.simple_keywords import BasisSet, Dft, Approximation, Grid
from opi.input.blocks import BlockMethod
from opi.input.structures.structure import Structure
from opi.utils.units import AU_TO_EV

# > For plotting results visualization of molecules
import matplotlib.pyplot as plt
import seaborn as sns
import py3Dmol


## Step 2: Working Directory and Conversion Factor

We define a subfolder `RUN` in which the actual ORCA calculations will take place. Also, we define a conversion factor, since we want the resulting interaction energies in kcal/mol for better interpretability.

In [2]:
# > Calculation is performed in `RUN`
working_dir = Path("RUN")
# > The `working_dir`is automatically (re-)created
#shutil.rmtree(working_dir, ignore_errors=True)
#working_dir.mkdir()
# > Conversion factor for atomic units to kcal/mol
unit_conversion = 627.509 

## Step 3: Setup the Input Structure

As an example we will look at the interaction of methanol with three water molecules. The 3D structure in Cartesian coordinates is defined and visualized:


In [3]:
# > Define cartesian coordinates in Angstroem as python string 
# > DOBNA
xyz_data = """\
32

B     -0.0710665      1.7160187     -0.1759734
C      1.3309628      2.3624373     -0.1437441
C      1.6317096      3.7406407     -0.1905028
H      0.8169200      4.4533881     -0.2244143
C      2.9284376      4.2214613     -0.1691227
H      3.1156876      5.2894216     -0.2036489
C      3.9980749      3.3237803     -0.0979803
H      5.0194699      3.6907928     -0.0862525
C      3.7599281      1.9640841     -0.0256833
H      4.5694543      1.2464764      0.0495033
C      2.4425340      1.4993452     -0.0351088
C     -1.4827289      2.3005905     -0.3997858
C     -2.5806992      1.4283572     -0.2386211
C     -3.9050138      1.8479851     -0.3859375
H     -4.7032894      1.1291294     -0.2377933
C     -4.1640536      3.1615282     -0.7297411
H     -5.1908967      3.4919035     -0.8502889
C     -3.1084478      4.0546959     -0.9370304
H     -3.3120915      5.0796347     -1.2282782
C     -1.8045917      3.6230759     -0.7733224
H     -1.0008613      4.3248250     -0.9593634
C     -0.0593280      0.2180520      0.0570007
C     -1.2484129     -0.5182560      0.1664370
C     -1.2595900     -1.8909490      0.3853786
H     -2.2013658     -2.4205794      0.4665236
C     -0.0376612     -2.5496956      0.4874103
H     -0.0292652     -3.6221042      0.6541803
C      1.1738162     -1.8724817      0.3816589
H      2.1237635     -2.3872862      0.4629659
C      1.1411409     -0.4984565      0.1734895
O     -2.4626872      0.0953782      0.0691587
O      2.3456514      0.1342264      0.0768764\n
"""
# > Visualize the input structure
view = py3Dmol.view(width=400, height=400)
view.addModel(xyz_data, 'xyz')
view.setStyle({}, {'stick': {}, 'sphere': {'scale': 0.3}})
view.zoomTo()
view.show()

# > Write the input structure to a file
with open(working_dir / "struc.xyz","w") as f:
    f.write(xyz_data)
# > Read structure into object
structure = Structure.from_xyz(working_dir / "struc.xyz")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Step 4: OT Preparation

First we run the calculation for the neutral species, for the cation, and for the anion. Then we evaluate the initial value for our function to minimize.

In [4]:
def setup_calc(basename : str, working_dir: Path, structure: Structure, omega: float, charge: int = 0, mult: int = 1, ncores: int = 4) -> Calculator:
    # > Set up a Calculator object
    omega = round(omega, 2)
    calc_basename = f"{basename}_{omega}"
    calc = Calculator(basename=calc_basename, working_dir=working_dir)
    # > Assign structure to calculator
    calc.structure = copy.deepcopy(structure)
    calc.structure.charge = charge
    calc.structure.multiplicity = mult

    # > Define a simple keyword list for the ORCA calculation
    sk_list = [
    Dft.WB97M_V,
    BasisSet.DEF2_SVP,
    Grid.DEFGRID2,
    Approximation.RIJCOSX
    ]

    # use simple keywords in calculator
    calc.input.add_simple_keywords(*sk_list)

    # use block to adjust omega range-separation parameter
    calc.input.add_blocks(BlockMethod(rangesepmu=omega))
    
    # Define number of CPUs for the calcualtion
    calc.input.ncores = ncores # > 4 CPUs for this ORCA run

    return calc

calc_neutral = setup_calc("neutral", working_dir, structure, 0.3, 0, 1)
calc_cation = setup_calc("cation", working_dir, structure, 0.3, 1, 2)
calc_anion = setup_calc("anion", working_dir, structure, 0.3, -1, 2)

Then we run the calculation with `run_calc` and obtain the output:

In [5]:
def run_calc(calc: Calculator, do_print: bool = False) -> Output:
    # > Write the ORCA input file
    calc.write_input()
    # > Run the ORCA calculation
    if (do_print): print(f"Running ORCA calculation {calc.basename} ...", end="")
    calc.run()
    if (do_print): print("   Done")

    # > Get the output object
    output = calc.get_output()
    
    return output

output_neutral = run_calc(calc_neutral, do_print=True)
output_cation = run_calc(calc_cation, do_print=True)
output_anion = run_calc(calc_anion, do_print=True)

Running ORCA calculation neutral_0.3 ...   Done
Running ORCA calculation cation_0.3 ...   Done
Running ORCA calculation anion_0.3 ...   Done


The successful calculation has to be checked and parsed, which is done in `check_and_parse_output`:

In [6]:
def check_and_parse_output(output: Output):
    # > Check for proper termination of ORCA
    status = output.terminated_normally()
    if not status:
        # > ORCA did not terminate normally
        raise RuntimeError(f"ORCA did not terminate normally, see output file: {output.get_outfile()}")
    else:
        # > ORCA did terminate normally so we can parse the output
        output.parse()

    # Now check for convergence of the SCF
    if not output.results_properties.geometries[0].single_point_data.converged:
        raise RuntimeError("SCF DID NOT CONVERGE")
    
check_and_parse_output(output_neutral)
check_and_parse_output(output_cation)
check_and_parse_output(output_anion)

# Objective function to minimize

First we need to define a function to obtain the energy of the HOMO for RHF calculations and the SOMO for UHF calculations:

In [7]:
def eval_j(output_cation, output_neutral, output_anion) -> float:
    """Returns j²(omega) function"""
    IP = (output_cation.get_final_energy() - output_neutral.get_final_energy()) * AU_TO_EV
    EA = (output_neutral.get_final_energy() - output_anion.get_final_energy()) * AU_TO_EV
    homo_neutral = output_neutral.get_homo().orbitalenergy * AU_TO_EV
    homo_anion = output_anion.get_homo().orbitalenergy * AU_TO_EV
    j = (homo_neutral+IP)**2 + (homo_anion+EA)**2
    return j

j = eval_j(output_cation, output_neutral, output_anion)
print(f"Initial J² value is {j:.4f}")

Initial J² value is 0.1932


# Putting it together into an objective function

In [ ]:
def optimal_tuning(omega: float, working_dir: Path, structure: Structure) -> float:
    """Evaluate J²(omega)"""
    # Set up all calculations first
    calc_neutral = setup_calc("neutral", working_dir, structure, omega, 0, 1)
    calc_cation = setup_calc("cation", working_dir, structure, omega, 1, 2)
    calc_anion = setup_calc("anion", working_dir, structure, omega, -1, 2)

    # Run in parallel
    with ThreadPoolExecutor(max_workers=3) as executor:
        future_neutral = executor.submit(run_calc, calc_neutral)
        future_cation = executor.submit(run_calc, calc_cation)
        future_anion = executor.submit(run_calc, calc_anion)

        output_neutral = future_neutral.result()
        output_cation = future_cation.result()
        output_anion = future_anion.result()

    # Continue with parsing and evaluation
    check_and_parse_output(output_neutral)
    check_and_parse_output(output_cation)
    check_and_parse_output(output_anion)

    j = eval_j(output_cation, output_neutral, output_anion)
    print(f"For omega {omega:.2f} the J² value is: {j:.4f}.")
    return j

# Optimize the function with scipy

In [9]:
fixed_function = partial(optimal_tuning, working_dir=working_dir, structure=structure)
result = minimize_scalar(fixed_function, bounds=(0.010, 0.45), method='bounded', options={'xatol': 1e-2})
print(f"omega: {result.x:.2f} J²(omega) {result.fun:.4f}")

For omega 0.178065 the J² value is: 0.006007.
For omega 0.281935 the J² value is: 0.140105.


KeyboardInterrupt: 

# Scan the function

In [ ]:
omega_vals = np.linspace(0.010, 0.450, 45)
j_vals = [optimal_tuning(omega,working_dir=working_dir,structure=structure) for omega in omega_vals]

# Plot the function

In [ ]:
plt.plot(omega_vals, j_vals)
plt.xlabel("omega")
plt.ylabel("J**2")
plt.title("Scan of omega from 0.01 to 0.45")
plt.grid(True)
plt.show()